###  Exercise 1: A small voice-based assistant
- Combine **Whisper** and **GPT** in a single pipeline:
  1. Record a voice message asking a question 
  2. Transcribe it using Whisper.
  3. Send the transcribed text to `openai.ChatCompletion.create(...)`.
  4. use TTS to speak the response.

In [9]:
%pip install requests
%pip install numpy pandas matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install openai python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Install required packages
# Run this in terminal or uncomment the line below
# !pip install openai python-dotenv

import openai
import os
from pathlib import Path
import io
from IPython.display import Audio, display
import base64

# For environment variables (recommended for API key management)
from dotenv import load_dotenv
load_dotenv()

# Set up OpenAI client
client = openai.OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)


print("OpenAI client initialized successfully!") 

OpenAI client initialized successfully!


In [4]:
def text_to_speech_basic(text, voice="alloy", model="tts-1"):
    """
    Convert text to speech using OpenAI's TTS API
    
    Args:
        text (str): Text to convert to speech
        voice (str): Voice to use (alloy, echo, fable, onyx, nova, shimmer)
        model (str): Model to use (tts-1 or tts-1-hd)
    
    Returns:
        bytes: Audio data
    """
    try:
        # Make API call to OpenAI TTS
        response = client.audio.speech.create(
            model=model,
            voice=voice,
            input=text,
            response_format="mp3"
        )
        
        # Return the audio content
        return response.content
        
    except Exception as e:
        print(f"Error in TTS: {e}")
        return None
    
def speech_to_text_basic(audio_file_path):
    """
    Convert speech to text using OpenAI's Whisper API
    
    Args:
        audio_file_path (str): Path to audio file
    
    Returns:
        str: Transcribed text
    """
    try:
        # Open the audio file
        with open(audio_file_path, "rb") as audio_file:
            # Make API call to Whisper
            transcript = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file,
                response_format="text"
            )
        
        return transcript
        
    except Exception as e:
        print(f"Error in STT: {e}")
        return None


In [5]:
## STEP 1
###################################################
########### Generate speech from text #############
###################################################

sample_text = "What is the best treatment for a grade 2 sprain?"
audio_data = text_to_speech_basic(sample_text)

if audio_data:
    # Save to file
    with open("extra_step1_text_to_audio.mp3", "wb") as f:
        f.write(audio_data)
    
    # Display audio player in Jupyter
    display(Audio(audio_data, autoplay=False))
    print("Audio generated successfully!")
    print("STEP 1 OK")
else:
    print("Failed to generate audio")
    print("STEP 1 --KO--")

Audio generated successfully!
STEP 1 OK


In [6]:
## STEP 2
###################################################
###########  Transcribe it using Whisper###########
###################################################    
test_audio_file = "extra_step1_text_to_audio.mp3"
with open(test_audio_file, "wb") as f:
     f.write(audio_data)
    
print("Generated audio file:", test_audio_file)
    
# Now transcribe it back to text
transcribed_text = speech_to_text_basic(test_audio_file)
    
if transcribed_text:
    print("Transcribed text:", transcribed_text)
    print("STEP 2 OK")
else:
    print("Transcription failed")
    print("STEP 2 --KO--")

Generated audio file: extra_step1_text_to_audio.mp3
Transcribed text: What is the best treatment for a grade 2 sprain?

STEP 2 OK


In [7]:
## STEP 3
########################################################################################
########### Send the transcribed text to `openai.ChatCompletion.create(...)`############
########################################################################################

import os
from openai import OpenAI

# CLENT
client = OpenAI()

#  API CALL
response = client.chat.completions.create(
    model="gpt-4o-mini",  # O el modelo que prefieras usar
    messages=[
        {"role": "system", "content": "You are a helpful assistant that resolve the user's request."},
        {"role": "user", "content": f"Please summarize the following transcription: {transcribed_text}"}
    ],
    temperature=0
)

# Así extraes el resultado en la versión nueva
final_result = response.choices[0].message.content
print(final_result)
print("STEP 3 OK")

The best treatment for a grade 2 sprain typically includes rest, ice, compression, and elevation (RICE). It's important to avoid putting weight on the injured area and to use ice to reduce swelling. Compression with a bandage can help stabilize the injury, and elevating the affected limb can further decrease swelling. Over-the-counter pain relievers may also be used to manage pain. In some cases, physical therapy may be recommended to aid recovery and restore strength and flexibility. If symptoms persist, consulting a healthcare professional is advisable.
STEP 3 OK


In [8]:
## STEP 4
#######################################################
########### use TTS to speak the response. ############
#######################################################

# Generate speech from the final result
final_audio = text_to_speech_basic(final_result)
if final_audio:
    # Save to file
    with open("extra_step4_final_result.mp3", "wb") as f:
        f.write(final_audio)
    
    # Display audio player in Jupyter
    display(Audio(final_audio, autoplay=False))
    print("Final audio generated successfully!")
    print("STEP 4 OK")

Final audio generated successfully!
STEP 4 OK


## Exercise 2: 

- Input: audio
- transcribe the audio
- Summarize it
- translate the summarization
- use TTS to speak the final results